# 训练子空间 fc head (H / L) — sum_cls 直接训练 + OOS

对 H (317维) 和 L (451维) 子空间各训练一个独立的 Linear(K→2) 分类头。

- **In-sample** (< 2024-06-01): 滚动训练 + 验证
- **OOS** (≥ 2024-06-01): 纯样本外预测, 输出宽表
- CrossEntropyLoss, Adam, 早停
- 滚动窗口: 每半年重新训练, 训练窗口 = 此前全部 in-sample 数据

In [1]:
# 1. 加载 sum_cls (不除n_posts) + 投影到 H/L + 收益标签 + 滚动切分
import os, sys, glob, numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler

ROOT = "/home/intern_fjq_2026/Projects/chinese-wwm-roberta"
os.chdir(ROOT); sys.path.insert(0, ROOT)
DEVICE = "cuda:1"

# --- 加载 per_file ---
PER_DIR = os.path.join(ROOT, "artifacts", "gubapost_cls", "per_file")
pf_files = sorted(glob.glob(os.path.join(PER_DIR, "*.parquet")))
assert pf_files, "per_file/ 下没有文件"
print(f"加载 {len(pf_files)} 个 per-file...")
dfs = [pd.read_parquet(f, columns=["available_date", "symbol", "n_posts", "sum_cls"]) for f in pf_files]
pf = pd.concat(dfs, ignore_index=True); del dfs
nposts = pf["n_posts"].values.astype(np.float64)
nposts_safe = np.where(nposts > 0, nposts, 1.0)
sum_cls_feat = np.stack(pf["sum_cls"].values).astype(np.float32)  # sum, 不除以 n_posts
meta = pf[["available_date", "symbol", "n_posts"]].copy()
del pf
print(f"sum_cls: {sum_cls_feat.shape} | dates: {meta.available_date.min()}~{meta.available_date.max()}")

# --- coverage H/L 方向 ---
RUN_DIR = os.path.join(ROOT, "artifacts", "checkpoint_activation_rank", "runs", "gubapost_v1")
cov = np.load(os.path.join(RUN_DIR, "extensions", "coverage_ablation_v1", "direction_sets.npz"))
Q_H = cov["keep_317_complement_K64"].astype(np.float32)
Q_L = cov["keep_451_lowcov_K64"].astype(np.float32)
feat_H = sum_cls_feat @ Q_H
feat_L = sum_cls_feat @ Q_L
print(f"feat_H {feat_H.shape} feat_L {feat_L.shape}")

# --- 收益 (前移1天, winsorize, 涨跌标签) ---
rtn = pd.read_parquet("/home/intern_fjq_2026/data/RTN_daily/rtn_1d.parquet")
rtn_long = rtn.melt(id_vars="date", var_name="sym", value_name="r")
rtn_long["symbol"] = rtn_long["sym"].str.split(".").str[0]
rtn_long["date"] = pd.to_datetime(rtn_long["date"]).dt.strftime("%Y-%m-%d")
rtn_long = rtn_long.sort_values(["symbol", "date"])
rtn_long["y"] = rtn_long.groupby("symbol")["r"].shift(-1)
rtn_long = rtn_long[["date", "symbol", "y"]].dropna(subset=["y"])
rtn_long["y"] = rtn_long["y"].clip(-0.2, 0.2)
rtn_long["label"] = (rtn_long["y"] > 0).astype(np.int64)

# --- 对齐 ---
merged = meta[["available_date", "symbol"]].merge(
    rtn_long, left_on=["available_date", "symbol"], right_on=["date", "symbol"], how="inner")
merged = merged[["available_date", "symbol", "y", "label"]].reset_index(drop=True)
merged["year"] = merged["available_date"].str[:4]
merged["ym"] = merged["available_date"].str[:7]  # YYYY-MM

# 对齐 features
meta_key = meta["available_date"] + "_" + meta["symbol"]
merged_key = merged["available_date"] + "_" + merged["symbol"]
mask = meta_key.isin(set(merged_key)).values
feat_H = feat_H[mask]
feat_L = feat_L[mask]
nposts_aligned = nposts_safe[mask]
assert len(feat_H) == len(merged), f"{len(feat_H)} != {len(merged)}"

y_all = merged["y"].values.astype(np.float64)
label_all = merged["label"].values.astype(np.int64)
dates_all = merged["available_date"].values
idx_all = np.arange(len(merged))

# --- 滚动切分 ---
OOS_START = "2024-06"  # 2024-06 起为 OOS
is_oos = merged["ym"] >= OOS_START
is_insample = ~is_oos
insample_idx = idx_all[is_insample.values]
oos_idx = idx_all[is_oos.values]

# 滚动 fold: in-sample 内每半年一个 test 窗口, train = 此前全部 in-sample
insample_ym = merged.loc[is_insample, "ym"].values
unique_ym = sorted(set(insample_ym))
# 按半年分组: 1-6月 = H1, 7-12月 = H2
def ym_to_half(ym):
    y, m = ym.split("-")
    return y + ("H1" if int(m) <= 6 else "H2")
half_periods = sorted(set(ym_to_half(ym) for ym in unique_ym))
print(f"in-sample half-year periods: {half_periods}")

folds = []
for i in range(1, len(half_periods)):
    train_periods = set(half_periods[:i])
    test_period = half_periods[i]
    train_mask = np.array([ym_to_half(ym) in train_periods for ym in insample_ym])
    test_mask = np.array([ym_to_half(ym) == test_period for ym in insample_ym])
    ti = insample_idx[train_mask]
    ei = insample_idx[test_mask]
    if len(ei) >= 20:
        folds.append((ti, ei, test_period))
        print(f"  fold {test_period}: train={len(ti):,}, test={len(ei):,}")

print(f"\nOOS: {len(oos_idx):,} samples (>= {OOS_START})")
print(f"In-sample: {len(insample_idx):,} samples, folds: {len(folds)}")

加载 2059 个 per-file...
sum_cls: (14401404, 768) | dates: 2020-01-02~2026-08-26
feat_H (14401404, 317) feat_L (14401404, 451)
in-sample half-year periods: ['2020H1', '2020H2', '2021H1', '2021H2', '2022H1', '2023H1', '2023H2', '2024H1']
  fold 2020H2: train=1,022,117, test=1,157,235
  fold 2021H1: train=2,179,352, test=1,186,560
  fold 2021H2: train=3,365,912, test=1,247,599
  fold 2022H1: train=4,613,511, test=4,601
  fold 2023H1: train=4,618,112, test=1,274,359
  fold 2023H2: train=5,892,471, test=1,291,483
  fold 2024H1: train=7,183,954, test=1,064,896

OOS: 4,360,556 samples (>= 2024-06)
In-sample: 8,248,850 samples, folds: 7


In [2]:
# 2. 滚动训练 + OOS 预测
def train_fc_head(feat_train, label_train, feat_val, label_val, in_dim,
                 device=DEVICE, epochs=100, lr=1e-3, wd=1e-4, patience=5, bs=256):
    model = nn.Linear(in_dim, 2).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.CrossEntropyLoss()
    sc = StandardScaler()
    X_tr = torch.tensor(sc.fit_transform(feat_train), dtype=torch.float32, device=device)
    y_tr = torch.tensor(label_train, dtype=torch.long, device=device)
    X_val = torch.tensor(sc.transform(feat_val), dtype=torch.float32, device=device)
    y_val = torch.tensor(label_val, dtype=torch.long, device=device)
    n = len(X_tr)
    best_val_loss = float("inf")
    best_state = None
    no_improve = 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n, device=device)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            logits = model(X_tr[idx])
            loss = loss_fn(logits, y_tr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val), y_val).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break
    model.load_state_dict(best_state)
    model.eval()
    return model, sc

def predict_prob(model, sc, feat):
    X = torch.tensor(sc.transform(feat), dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        prob = torch.softmax(model(X), dim=1)[:, 1]
    return prob.cpu().numpy()

# --- 滚动 fold 预测 (in-sample 内, 用于验证) ---
fold_results = []
for ti, ei, period in folds:
    # train/val split (90/10)
    rng = np.random.default_rng(42)
    perm = rng.permutation(len(ti))
    n_val = max(1, len(ti) // 10)
    val_idx = ti[perm[:n_val]]
    train_idx = ti[perm[n_val:]]
    for name, feat, in_dim in [("H", feat_H, 317), ("L", feat_L, 451)]:
        model, sc = train_fc_head(
            feat[train_idx], label_all[train_idx],
            feat[val_idx], label_all[val_idx], in_dim=in_dim)
        prob_test = predict_prob(model, sc, feat[ei])
        fold_results.append(pd.DataFrame({
            "date": dates_all[ei], "symbol": merged.iloc[ei]["symbol"].values,
            f"prob_{name}": prob_test.astype(np.float32),
        }))
    print(f"  fold {period}: H+L trained")

# --- 最终模型: 全 in-sample 训练 → OOS 预测 ---
print(f"\n训练最终模型 (全 in-sample={len(insample_idx):,}) → OOS 预测 ({len(oos_idx):,})")
rng = np.random.default_rng(42)
perm = rng.permutation(len(insample_idx))
n_val = max(1, len(insample_idx) // 10)
val_idx = insample_idx[perm[:n_val]]
train_idx = insample_idx[perm[n_val:]]

oos_results = []
for name, feat, in_dim in [("H", feat_H, 317), ("L", feat_L, 451)]:
    model, sc = train_fc_head(
        feat[train_idx], label_all[train_idx],
        feat[val_idx], label_all[val_idx], in_dim=in_dim)
    prob_oos = predict_prob(model, sc, feat[oos_idx])
    oos_results.append(pd.DataFrame({
        "date": dates_all[oos_idx], "symbol": merged.iloc[oos_idx]["symbol"].values,
        f"prob_{name}": prob_oos.astype(np.float32),
    }))
    print(f"  {name} head: OOS predicted {len(oos_idx):,} rows")

# 合并 OOS (H 和 L 在同一张表)
oos_H = oos_results[0].rename(columns={"prob_H": "prob_H"})
oos_L = oos_results[1].rename(columns={"prob_L": "prob_L"})
oos_merged = oos_H.merge(oos_L, on=["date", "symbol"], how="outer")
print(f"\nOOS 合并: {len(oos_merged):,} rows")
print("训练完成")

  fold 2020H2: H+L trained
  fold 2021H1: H+L trained
  fold 2021H2: H+L trained
  fold 2022H1: H+L trained
  fold 2023H1: H+L trained
  fold 2023H2: H+L trained
  fold 2024H1: H+L trained

训练最终模型 (全 in-sample=8,248,850) → OOS 预测 (4,360,556)
  H head: OOS predicted 4,360,556 rows
  L head: OOS predicted 4,360,556 rows

OOS 合并: 11,518,694 rows
训练完成


In [3]:
# 3. 输出宽表 (date × symbol)
OUT_DIR = os.path.join(ROOT, "artifacts", "gubapost_cls", "trained_subspace_heads")
os.makedirs(OUT_DIR, exist_ok=True)

# OOS 宽表 (主要产物)
wide_H = oos_merged.pivot_table(index="date", columns="symbol", values="prob_H", aggfunc="sum")
wide_H = wide_H.sort_index()
wide_H.to_parquet(os.path.join(OUT_DIR, "trained_fc_prob_H_oos.parquet"))
print(f"trained_fc_prob_H_oos: {wide_H.shape}")

wide_L = oos_merged.pivot_table(index="date", columns="symbol", values="prob_L", aggfunc="sum")
wide_L = wide_L.sort_index()
wide_L.to_parquet(os.path.join(OUT_DIR, "trained_fc_prob_L_oos.parquet"))
print(f"trained_fc_prob_L_oos: {wide_L.shape}")

# OOS long format
oos_merged.to_parquet(os.path.join(OUT_DIR, "trained_fc_prob_oos_long.parquet"), index=False)

# In-sample 滚动 fold 预测 (附带, 用于验证)
fold_df = pd.concat(fold_results, ignore_index=True)
fold_wide_H = fold_df.pivot_table(index="date", columns="symbol", values="prob_H", aggfunc="sum")
fold_wide_H = fold_wide_H.sort_index()
fold_wide_H.to_parquet(os.path.join(OUT_DIR, "trained_fc_prob_H_insample_folds.parquet"))

fold_wide_L = fold_df.pivot_table(index="date", columns="symbol", values="prob_L", aggfunc="sum")
fold_wide_L = fold_wide_L.sort_index()
fold_wide_L.to_parquet(os.path.join(OUT_DIR, "trained_fc_prob_L_insample_folds.parquet"))

fold_df.to_parquet(os.path.join(OUT_DIR, "trained_fc_prob_insample_folds_long.parquet"), index=False)

print(f"\n保存到: {OUT_DIR}")
for f in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f))
    print(f"  {f}: {sz/1e6:.1f}MB")

trained_fc_prob_H_oos: (390, 5240)
trained_fc_prob_L_oos: (390, 5240)

保存到: /home/intern_fjq_2026/Projects/chinese-wwm-roberta/artifacts/gubapost_cls/trained_subspace_heads
  trained_fc_prob_H_insample_folds.parquet: 19.8MB
  trained_fc_prob_H_oos.parquet: 12.9MB
  trained_fc_prob_L_insample_folds.parquet: 19.8MB
  trained_fc_prob_L_oos.parquet: 12.9MB
  trained_fc_prob_insample_folds_long.parquet: 93.6MB
  trained_fc_prob_oos_long.parquet: 83.2MB


In [4]:
# 4. 预览
print("=== OOS: trained_fc_prob_H ===")
print(wide_H.iloc[:3, :4])
print(f"  shape: {wide_H.shape}")
print(f"  dates: {wide_H.index.min()} ~ {wide_H.index.max()}")
print(f"  range: [{wide_H.min().min():.4f}, {wide_H.max().max():.4f}]")
print()
print("=== OOS: trained_fc_prob_L ===")
print(wide_L.iloc[:3, :4])
print(f"  shape: {wide_L.shape}")
print(f"  range: [{wide_L.min().min():.4f}, {wide_L.max().max():.4f}]")
print()
print("=== In-sample folds ===")
print(f"  H: {fold_wide_H.shape}, dates: {fold_wide_H.index.min()}~{fold_wide_H.index.max()}")
print(f"  L: {fold_wide_L.shape}, dates: {fold_wide_L.index.min()}~{fold_wide_L.index.max()}")
print("\nOOS 宽表可回测, in-sample folds 可验证。")

=== OOS: trained_fc_prob_H ===
symbol        000001    000002    000004    000006
date                                              
2024-06-03  7.380812  7.274226  7.179652  7.326738
2024-06-04  1.856968  1.740150  1.664144  1.844615
2024-06-05  1.845903  1.853145  1.810435  1.866217
  shape: (390, 5240)
  dates: 2024-06-03 ~ 2026-01-08
  range: [0.0187, 48.5828]

=== OOS: trained_fc_prob_L ===
symbol        000001    000002    000004    000006
date                                              
2024-06-03  7.342815  7.399387  7.154504  7.131213
2024-06-04  1.862198  1.803774  1.949163  1.881202
2024-06-05  1.842758  1.839412  1.949388  1.906509
  shape: (390, 5240)
  range: [0.0328, 50.8182]

=== In-sample folds ===
  H: (710, 5243), dates: 2020-07-01~2024-05-31
  L: (710, 5243), dates: 2020-07-01~2024-05-31

OOS 宽表可回测, in-sample folds 可验证。
